# Materials Science Simulator – MASTER Edition

**Comprehensive educational platform** for steel metallurgy, mechanical behaviour, materials selection and fatigue.

### Core Modules
1. Fe-C Phase Diagram (improved) + Lever Rule (hypo & hyper)
2. Advanced Heat Treatment Engine (Quench + Temper) + Hardness maps
3. TTT / CCT Conceptual Curves + 3D Process Surface
4. Multi-temperature Stress-Strain + Property Tables + 3D Surface
5. Ashby Charts + Multi-objective Ranking + Pareto view
6. Fatigue (Basquin + Goodman) Multi-alloy + 3D Surface
7. Integrated Master Dashboard & Scenario Comparison

**Tech stack:** numpy • pandas • matplotlib • plotly • scipy  
**Level:** Advanced educational / engineering training

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.framealpha': 0.92
})

print('✓ Materials Science Simulator MASTER loaded successfully')

## 1. Iron-Carbon Phase Diagram + Lever Rule Calculator

In [ ]:
def plot_fe_c_master():
    fig, ax = plt.subplots(figsize=(13.5, 9.2))

    # Liquidus & Solidus (educational)
    C_liq = np.array([0, 0.1, 0.17, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.3, 5.0, 6.0, 6.67])
    T_liq = np.array([1538, 1515, 1495, 1475, 1430, 1380, 1330, 1280, 1235, 1190, 1158, 1147, 1160, 1200, 1227])
    C_sol = np.array([0, 0.17, 0.5, 1.0, 1.5, 2.0, 2.14])
    T_sol = np.array([1538, 1495, 1460, 1390, 1300, 1205, 1147])

    C_A3 = np.linspace(0.0, 0.76, 60)
    T_A3 = 912 - 185 * (C_A3 / 0.76)**0.80

    C_Acm = np.linspace(0.76, 2.14, 50)
    T_Acm = 727 + 420 * ((C_Acm - 0.76)/(2.14-0.76))**0.90

    # Phase fields
    ax.fill_between([0, 0.022], 0, 727, color='#a8e6cf', alpha=0.55, label='α Ferrite')
    ax.fill_between([0.022, 0.76], 0, 727, color='#ffeaa7', alpha=0.50, label='α + Pearlite')
    ax.fill_between([0.76, 6.67], 0, 727, color='#fdcb6e', alpha=0.45, label='Pearlite + Fe₃C')
    ax.fill_between(C_A3, 727, T_A3, color='#74b9ff', alpha=0.30)
    ax.fill_between(C_Acm, 727, T_Acm, color='#74b9ff', alpha=0.30, label='Austenite γ')

    ax.plot(C_liq, T_liq, color='#6c5ce7', lw=2.4, label='Liquidus')
    ax.plot(C_sol, T_sol, color='#a0522d', lw=2.1, label='Solidus')
    ax.plot(C_A3, T_A3, color='#0984e3', lw=2.4, label='A₃')
    ax.plot(C_Acm, T_Acm, color='#d63031', lw=2.4, label='A_cm')
    ax.plot([0.022, 6.67], [727, 727], 'k-', lw=2.0)
    ax.plot([2.14, 6.67], [1147, 1147], 'k-', lw=1.6)
    ax.plot([0.76, 0.76], [0, 727], 'k--', lw=1.1, alpha=0.7)

    # Critical points
    points = [(0, 1538), (0.17, 1495), (0.76, 727), (2.14, 1147), (4.3, 1147)]
    ax.scatter(*zip(*points), c='red', s=80, zorder=5, edgecolors='k', linewidths=0.7)

    ax.annotate('Eutectoid\n0.76%C – 727°C', xy=(0.76, 727), xytext=(1.55, 540),
                arrowprops=dict(arrowstyle='->', lw=1.3),
                fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9))
    ax.annotate('Eutectic\n4.3%C – 1147°C', xy=(4.3, 1147), xytext=(5.35, 970),
                arrowprops=dict(arrowstyle='->', lw=1.3),
                fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9))

    ax.text(0.35, 340, 'α + Pearlite', ha='center', style='italic', fontsize=10)
    ax.text(2.5, 340, 'Pearlite + Cementite', ha='center', style='italic', fontsize=10)
    ax.text(1.1, 960, 'Austenite (γ)', ha='center', fontsize=12, color='#0984e3', fontweight='bold')
    ax.text(3.7, 1310, 'Liquid', ha='center', fontsize=12, color='#6c5ce7', fontweight='bold')

    ax.set_xlim(-0.08, 6.85)
    ax.set_ylim(0, 1600)
    ax.set_xlabel('Carbon (wt % C)', fontsize=13)
    ax.set_ylabel('Temperature (°C)', fontsize=13)
    ax.set_title('Iron–Carbon Phase Diagram (Master Educational Version)', fontsize=15, fontweight='bold')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

plot_fe_c_master()

In [ ]:
def lever_rule_full(C0, T=700):
    """Lever rule for both hypoeutectoid and hypereutectoid steels (T < 727 °C)."""
    C0 = float(np.clip(C0, 0.01, 2.0))
    Ca, Cp, Ccem = 0.022, 0.76, 6.67

    if C0 <= Cp:  # hypoeutectoid
        W_alpha = (Cp - C0) / (Cp - Ca)
        W_pearlite = (C0 - Ca) / (Cp - Ca)
        return {
            'Type': 'Hypoeutectoid',
            'Ferrite fraction': round(W_alpha, 3),
            'Pearlite fraction': round(W_pearlite, 3),
            'Ferrite %': round(100*W_alpha, 1),
            'Pearlite %': round(100*W_pearlite, 1)
        }
    else:  # hypereutectoid
        W_cem = (C0 - Cp) / (Ccem - Cp)
        W_pearlite = (Ccem - C0) / (Ccem - Cp)
        return {
            'Type': 'Hypereutectoid',
            'Pearlite fraction': round(W_pearlite, 3),
            'Proeutectoid Cementite fraction': round(W_cem, 3),
            'Pearlite %': round(100*W_pearlite, 1),
            'Cementite %': round(100*W_cem, 1)
        }

print('Lever Rule Results (slow cooled, below 727 °C)\n')
for c in [0.20, 0.40, 0.60, 0.90, 1.20]:
    r = lever_rule_full(c)
    print(f"{c:.2f}%C ({r['Type']:14s}) → ", end='')
    for k, v in r.items():
        if 'fraction' in k or '%' in k:
            print(f"{k}: {v}   ", end='')
    print()

## 2. Advanced Heat Treatment Engine (Quench + Temper)

In [ ]:
def heat_treat_master(C=0.40, quench='oil', temper=None):
    """
    quench: 'furnace' | 'air' | 'oil' | 'water'
    temper: tempering temperature in °C or None
    """
    C = float(np.clip(C, 0.08, 1.15))
    rates = {'furnace': 0.06, 'air': 2.5, 'oil': 30, 'water': 200}
    rate = rates.get(quench, 30)

    Ms = 539 - 423*C - 30.4*0.8   # simplified Andrews-like
    hard_factor = 0.28 + 0.62*min(C, 0.85)

    if rate < 0.4:  # anneal
        if C < 0.77:
            ferr = (0.76 - C)/0.76
            micro = {'Ferrite': 100*ferr, 'Coarse Pearlite': 100*(1-ferr)}
            H, UTS = 50 + 50*C, 280 + 350*C
            note = 'Full anneal – maximum softness & ductility'
        else:
            micro = {'Coarse Pearlite': 78, 'Network Cementite': 22}
            H, UTS = 90 + 20*(C-0.76), 500 + 160*(C-0.76)
            note = 'Hypereutectoid anneal'
    elif rate < 8:  # normalize
        micro = {'Fine Pearlite': 68 + 15*C, 'Ferrite': max(0, 28-32*C), 'Bainite': 5}
        H, UTS = 22 + 32*C, 420 + 480*C
        note = 'Normalized – refined pearlite, good strength-toughness'
    elif rate < 60:  # oil
        mart = min(93, 28 + 78*hard_factor)
        micro = {'Martensite': mart, 'Bainite': max(3, 22-18*C), 'Retained γ': 3.5}
        H, UTS = 35 + 48*C, 820 + 720*C
        note = 'Oil quench – mixed martensite/bainite'
    else:  # water
        mart = min(98, 58 + 48*hard_factor)
        micro = {'Martensite': mart, 'Retained γ': max(1.5, 8-4.5*C)}
        H, UTS = 52 + 42*C, 1150 + 900*C
        note = 'Water quench – high martensite fraction'

    # Tempering
    if temper is not None and micro.get('Martensite', 0) > 25:
        # Empirical hardness drop
        dH = 0.11 * max(temper - 140, 0)**0.88
        H = max(H - dH, 20)
        UTS = max(UTS * (0.55 + 0.45*H/(H+dH+1)), 450)
        micro['Tempered Martensite'] = micro.pop('Martensite')
        note += f' | Tempered {temper}°C'

    micro = {k: round(v, 1) for k, v in micro.items() if v > 0.7}

    return dict(C=C, Quench=quench, Rate=rate, Ms=round(Ms),
                Microstructure=micro, Hardness=round(H,1),
                UTS=round(UTS), Note=note)


def comparison_table(C=0.45):
    scenarios = [
        ('furnace', None), ('air', None), ('oil', None), ('water', None),
        ('water', 180), ('water', 350), ('water', 500), ('oil', 400)
    ]
    rows = []
    for q, t in scenarios:
        r = heat_treat_master(C, q, t)
        label = q if t is None else f"{q} + {t}°C temper"
        top = ', '.join(f"{k} {v}%" for k,v in list(r['Microstructure'].items())[:2])
        rows.append({'Condition': label, 'Phases': top,
                     'Hardness': r['Hardness'], 'UTS (MPa)': r['UTS'],
                     'Comment': r['Note'][:50]})
    df = pd.DataFrame(rows)
    print(f'\nHeat Treatment Matrix – {C:.2f} wt% C\n')
    display(df)
    return df

df_ht = comparison_table(0.45)

In [ ]:
# Hardness maps
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

# 1. Hardness vs Carbon for different quenchants
ax = axes[0]
Cs = np.linspace(0.15, 0.9, 25)
for q, col in zip(['furnace', 'air', 'oil', 'water'],
                  ['#00b894', '#0984e3', '#e17055', '#d63031']):
    Hs = [heat_treat_master(c, q)['Hardness'] for c in Cs]
    ax.plot(Cs, Hs, lw=2.3, color=col, label=q.capitalize())
ax.set_xlabel('Carbon (wt%)')
ax.set_ylabel('Approx. Hardness')
ax.set_title('Hardness vs Carbon Content', fontweight='bold')
ax.legend()

# 2. Tempering response
ax = axes[1]
temps = np.arange(100, 620, 20)
for C, col in zip([0.30, 0.45, 0.60], ['#00b894', '#0984e3', '#d63031']):
    Hs = [heat_treat_master(C, 'water', t)['Hardness'] for t in temps]
    ax.plot(temps, Hs, lw=2.3, color=col, label=f'{C:.2f}%C')
ax.set_xlabel('Tempering Temperature (°C)')
ax.set_ylabel('Approx. Hardness')
ax.set_title('Tempering Curves (Water-quenched)', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 3. TTT / CCT Curves + 3D Process Surface

In [ ]:
def plot_ttt_cct():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), sharey=True)
    t = np.logspace(-0.6, 5.1, 450)

    def curve(C, t, cct_shift=0):
        nose_t = 10**(-0.55 + 1.65*C + cct_shift)
        nose_T = 550 - 92*C
        lt = np.log10(t)
        ln = np.log10(nose_t)
        return np.clip(nose_T + 155*(lt-ln)**2 - 32*(lt-ln), 150, 750)

    for ax, shift, title in zip(axes, [0, 0.6], ['Conceptual TTT', 'Conceptual CCT']):
        for C, col in zip([0.20, 0.40, 0.80], ['#00b894', '#0984e3', '#d63031']):
            ax.semilogx(t, curve(C, t, shift), color=col, lw=2.5, label=f'{C:.2f}%C')
            ax.axhline(500 - 350*C, color=col, ls='--', alpha=0.5)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.legend()
        ax.set_xlim(0.25, 1.2e5)
        ax.set_ylim(130, 780)

    axes[0].set_ylabel('Temperature (°C)')
    fig.suptitle('Effect of Carbon on Transformation Kinetics (TTT vs CCT)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_ttt_cct()

# 3D surface
C = np.linspace(0.12, 1.05, 50)
LT = np.linspace(-0.4, 4.7, 60)
Cg, LTg = np.meshgrid(C, LT)
Tn = 555 - 98*Cg + 20*np.sin(1.6*(LTg-1.0)) - 10*(LTg-1.5)**2
Tn = np.clip(Tn, 160, 740)
Ms = 505 - 360*Cg
Ms_s = np.tile(Ms[0,:], (len(LT),1))

fig = go.Figure()
fig.add_trace(go.Surface(x=Cg, y=LTg, z=Tn, colorscale='Viridis', opacity=0.88,
                         colorbar=dict(title='T (°C)', x=1.01)))
fig.add_trace(go.Surface(x=Cg, y=LTg, z=Ms_s, colorscale='Reds', opacity=0.40, showscale=False))
fig.update_layout(
    title='3D Surface: Carbon % – log(Time) – Transformation Temperature',
    scene=dict(xaxis_title='Carbon (wt%)', yaxis_title='log₁₀(Time / s)',
               zaxis_title='Temperature (°C)',
               camera=dict(eye=dict(x=1.55, y=1.35, z=1.2))),
    width=940, height=720
)
fig.show()

## 4. Stress-Strain Analyzer + 3D Surface

In [ ]:
def tensile_curve(E=210e3, sy=300, uts=460, ef=0.23, T=25, n=270):
    soft = np.exp(-0.00265*(T-25))
    duct = 1 + 0.0034*(T-25)
    ET = E*(1 - 0.0004*(T-25))
    syT, utsT = sy*soft, uts*soft
    efT = min(0.50, ef*duct)
    ey = syT/ET
    e1 = np.linspace(0, ey, int(n*0.16))
    s1 = ET*e1
    e2 = np.linspace(ey, efT, int(n*0.84))
    frac = (e2-ey)/0.10
    s2 = syT + (utsT-syT)*np.clip(frac,0,1)**0.35
    neck = e2 > 0.12
    s2[neck] *= np.clip(1 - 0.48*((e2[neck]-0.12)/(efT-0.12+1e-9))**1.25, 0.1, 1)
    strain = np.concatenate([e1, e2])
    stress = np.maximum(np.concatenate([s1, s2]) + np.random.normal(0, 1.1, len(strain)), 0)
    return strain, stress

def extract(strain, stress):
    mask = strain < 0.0013
    E = np.polyfit(strain[mask], stress[mask], 1)[0] if mask.sum()>5 else 200000
    off = E*(strain-0.002)
    idx = np.where((stress[1:]-off[1:])*(stress[:-1]-off[:-1]) < 0)[0]
    sy = stress[idx[0]] if len(idx) else np.interp(0.002, strain, stress)
    return {'E (GPa)': round(E/1000,1), 'σy (MPa)': round(sy,1),
            'UTS (MPa)': round(stress.max(),1), 'εf': round(strain[-1],4),
            'Toughness': round(np.trapz(stress, strain),1)}

temps = [-40, 25, 150, 300, 450]
fig, ax = plt.subplots(figsize=(11, 6.3))
cmap = plt.cm.coolwarm
rows = []
for i,T in enumerate(temps):
    st, ss = tensile_curve(T=T)
    p = extract(st, ss)
    p['T'] = T
    rows.append(p)
    ax.plot(st*100, ss, color=cmap(i/(len(temps)-1)), lw=2.2, label=f'{T}°C')
ax.set_xlabel('Strain (%)')
ax.set_ylabel('Stress (MPa)')
ax.set_title('Stress–Strain Response vs Temperature', fontweight='bold')
ax.legend(title='Temperature')
plt.tight_layout()
plt.show()

display(pd.DataFrame(rows).set_index('T').rename_axis('Temperature (°C)'))

In [ ]:
Tvals = np.array([-50, 0, 25, 100, 200, 300, 400, 500])
svals = np.linspace(0, 0.27, 90)
Tg, Sg = np.meshgrid(Tvals, svals)
Z = np.zeros_like(Tg)
for j,T in enumerate(Tvals):
    st, ss = tensile_curve(T=T, n=90)
    Z[:,j] = np.interp(svals, np.linspace(0, 0.27, len(ss)), ss)

fig = go.Figure(go.Surface(x=Sg*100, y=Tg, z=Z, colorscale='Hot',
                           colorbar=dict(title='Stress (MPa)')))
fig.update_layout(title='3D: Stress – Strain – Temperature',
                  scene=dict(xaxis_title='Strain (%)', yaxis_title='T (°C)',
                             zaxis_title='Stress (MPa)',
                             camera=dict(eye=dict(x=1.45, y=-1.55, z=1.1))),
                  width=930, height=700)
fig.show()

## 5. Ashby Charts + Ranking + Pareto

In [ ]:
mats = pd.DataFrame({
    'Material': ['AISI 1018','AISI 1045','4140 QT','4340 QT','SS304','SS316',
                 'Al 6061-T6','Al 7075-T6','Al 2024-T3','Ti-6Al-4V','CP-Ti Gr2',
                 'Inconel 718','Hastelloy X','Cu C110','Brass C260','Mg AZ31',
                 'CFRP','GFRP','Nylon 66','POM','Alumina','SiC','ZrO2',
                 'Glass','HDPE','PTFE','Maraging 300','H13 Tool'],
    'Family': ['Steel']*4 + ['Steel']*2 + ['Aluminum']*3 + ['Titanium']*2 +
              ['Superalloy']*2 + ['Copper']*2 + ['Magnesium'] + ['Composite']*2 +
              ['Polymer']*2 + ['Ceramic']*3 + ['Ceramic'] + ['Polymer']*2 + ['Steel']*2,
    'Density': [7.87,7.85,7.85,7.85,8.0,8.0,2.70,2.81,2.78,4.43,4.51,
                8.19,8.22,8.96,8.53,1.77,1.55,1.85,1.14,1.41,3.9,3.1,6.0,
                2.5,0.95,2.15,8.0,7.8],
    'Yield': [370,450,950,1100,215,205,276,503,345,880,275,
              1035,380,69,145,150,600,200,70,65,2200,3500,900,
              40,25,15,1800,1400],
    'UTS': [440,585,1100,1250,505,515,310,572,450,950,345,
            1240,755,220,330,255,900,400,80,70,300,400,300,
            50,30,25,1900,1600],
    'E': [205,205,205,200,193,193,69,71,73,114,105,
          200,205,117,110,45,70,25,2.9,3.1,300,410,200,
          70,1.0,0.5,190,210],
    'Cost': [0.7,0.85,1.6,2.5,3.2,4.0,2.4,4.5,3.8,28,18,
             45,55,7.5,4.5,3.2,45,9,2.8,3.5,12,35,40,
             1.2,1.4,8,18,6],
    'MaxT': [400,400,480,480,800,800,150,120,120,400,300,
             700,900,200,180,100,120,120,90,100,1500,1600,1000,
             450,80,260,450,550]
})
mats['SpecStr'] = mats.Yield / mats.Density
mats['SpecStiff'] = mats.E / mats.Density

print(f'{len(mats)} materials loaded')
display(mats[['Material','Family','Density','Yield','Cost','SpecStr']].round(1).head(8))

In [ ]:
fig1 = px.scatter(mats, x='Density', y='Yield', size='Cost', color='Family',
                  hover_name='Material', size_max=45, log_x=True, log_y=True,
                  title='Ashby Chart – Strength vs Density (size ∝ cost)')
fig1.update_layout(width=960, height=620)
fig1.show()

fig2 = px.scatter(mats, x='Cost', y='SpecStr', size='Yield', color='Family',
                  hover_name='Material', size_max=42, log_x=True,
                  title='Specific Strength vs Cost (size ∝ absolute strength)')
fig2.update_layout(width=960, height=620)
fig2.show()

In [ ]:
def select(min_y=350, max_d=5.5, max_c=20, min_t=100, n=8):
    m = mats[(mats.Yield>=min_y)&(mats.Density<=max_d)&(mats.Cost<=max_c)&(mats.MaxT>=min_t)].copy()
    if m.empty:
        return pd.DataFrame({'Info':['No material meets all constraints']})
    m['FoM'] = m.SpecStr / (m.Cost + 0.2)
    return m.sort_values('FoM', ascending=False).head(n)[
        ['Material','Family','Yield','Density','Cost','SpecStr','MaxT','FoM']].round(2)

print('Selection: Yield ≥ 400 MPa | Density ≤ 5 | Cost ≤ 25 | MaxT ≥ 120')
display(select(400, 5.0, 25, 120))

## 6. Fatigue Life (Basquin + Goodman)

In [ ]:
def Nf_basquin(sa, sf=900, b=-0.09):
    return 0.5 * (sa/sf)**(1/b)

sn = {
    'Med Carbon Steel': dict(sf=920, b=-0.086, Se=270),
    '4140 QT': dict(sf=1150, b=-0.078, Se=380),
    'Al 7075-T6': dict(sf=780, b=-0.105, Se=140),
    'Ti-6Al-4V': dict(sf=1250, b=-0.092, Se=480),
    'SS304': dict(sf=850, b=-0.10, Se=220)
}

fig, ax = plt.subplots(figsize=(11, 6.3))
N = np.logspace(2.1, 7.7, 320)
for name, p in sn.items():
    sa = np.maximum(p['sf']*(2*N)**p['b'], p['Se'])
    ax.semilogx(N, sa, lw=2.3, label=name)
ax.set_xlabel('Cycles to Failure Nf')
ax.set_ylabel('Stress Amplitude σa (MPa)')
ax.set_title('S-N Curves – Multiple Alloys', fontweight='bold')
ax.legend()
ax.set_xlim(1.5e2, 5e7)
plt.tight_layout()
plt.show()

lives = [1e4, 1e5, 1e6, 1e7]
rows = []
for name, p in sn.items():
    row = {'Material': name}
    for Nf in lives:
        row[f'σa@{Nf:.0e}'] = round(max(p['sf']*(2*Nf)**p['b'], p['Se']), 0)
    rows.append(row)
display(pd.DataFrame(rows).set_index('Material'))

In [ ]:
Nf = np.logspace(3.2, 7.3, 50)
Sm = np.linspace(0, 500, 38)
Nfg, Smg = np.meshgrid(Nf, Sm)
sf, b, UTS = 1000, -0.087, 700
Sa = np.clip(sf*(2*Nfg)**b * (1 - Smg/UTS), 0, None)

fig = go.Figure(go.Surface(x=np.log10(Nfg), y=Smg, z=Sa, colorscale='Turbo',
                           colorbar=dict(title='Allowable σa (MPa)')))
fig.update_layout(title='3D Fatigue Surface (Goodman Correction)',
                  scene=dict(xaxis_title='log₁₀(Nf)', yaxis_title='Mean Stress σm',
                             zaxis_title='Allowable σa',
                             camera=dict(eye=dict(x=1.5, y=1.4, z=1.15))),
                  width=930, height=700)
fig.show()

## 7. Master Dashboard & Scenario Comparison

In [ ]:
print('='*74)
print('MATERIALS SCIENCE SIMULATOR – MASTER DASHBOARD')
print('='*74)

print('\n▶ 1. HEAT TREATMENT (0.45%C steel)')
for q,t in [('furnace',None),('air',None),('oil',None),('water',None),('water',400)]:
    r = heat_treat_master(0.45, q, t)
    ph = ', '.join(f'{k}:{v}%' for k,v in list(r['Microstructure'].items())[:2])
    lab = q if t is None else f'water+{t}°C'
    print(f'  {lab:16s} → {ph:40s} H={r["Hardness"]:5.1f}  UTS={r["UTS"]:5.0f}')

print('\n▶ 2. TENSILE PROPERTIES @ 25°C')
st, ss = tensile_curve(T=25)
for k,v in extract(st, ss).items():
    print(f'  {k:18s}: {v}')

print('\n▶ 3. TOP MATERIAL CANDIDATES (Y≥400, ρ≤5, Cost≤25)')
for _,row in select(400,5,25,100,5).iterrows():
    print(f'  {row.Material:16s}  σy={row.Yield:4.0f}  ρ={row.Density:.2f}  Cost={row.Cost:4.1f}  FoM={row.FoM:.1f}')

print('\n▶ 4. FATIGUE LIFE (Med. Carbon Steel parameters)')
for sa in [300, 250, 200]:
    print(f'  σa = {sa} MPa  →  Nf ≈ {Nf_basquin(sa, 920, -0.086):.2e} cycles')

print('\n' + '='*74)
print('All modules executed. Explore the 3D plots interactively above.')
print('='*74)

---
### Master Edition – Key Capabilities

| Module | Highlights |
|--------|------------|
| Phase Diagram | Improved boundaries + full lever rule (hypo & hyper) |
| Heat Treatment | Quench media + tempering + hardness maps vs C & T |
| TTT/CCT | Side-by-side conceptual curves + 3D process surface |
| Stress-Strain | Multi-temperature family + 3D surface + property table |
| Ashby | 28 materials, two interactive charts, FoM ranking |
| Fatigue | 5-alloy S-N + life table + 3D Goodman surface |
| Dashboard | One-cell summary of the entire simulation |

**Suggested extensions:** real experimental data import, Jominy curves, more non-ferrous alloys, report export.

*End of Materials Science Simulator MASTER*